In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import find_peaks, peak_widths, savgol_filter
from scipy.stats import binned_statistic
import pandas as pd
import h5py
from typing import Literal
import struct

plt.rcParams["figure.dpi"] = 500

In [ ]:
def plot_with_peaks(
    data,
    n_bins,
    prominence=None,
    xlabel="",
    title="",
    xlims=None,
    savgol_window=None,
    savgol_strength=1.0,
    clip_counts=None,
    stacked=False,
    legend=None,
    bin_range=None,
    text_size=5,
    divide=1.0,
    rng_weight=0.0,
):
    if not stacked:
        signals = [data]
    else:
        signals = data

    bottom = np.zeros(n_bins)

    for signal in signals:
        counts, bins = np.histogram(signal, bins=n_bins, density=False, range=bin_range)
        counts = counts / divide
        if savgol_window is not None:
            background = savgol_filter(counts, window_length=savgol_window, polyorder=3)
            counts = counts - background * savgol_strength

        if clip_counts is not None:
            counts[counts > clip_counts] = clip_counts

        bin_width = np.diff(bins).mean()
        bin_centers = (bins[:-1] + bins[1:]) / 2.0
        height_scale = counts.max()

        plt.bar(bin_centers, counts, width=bin_width, align="center", bottom=bottom)
        plt.xlim(xlims)
        bottom += counts
        bottom = np.round(bottom, 10)
        if prominence is not None:
            peaks, props = find_peaks(counts, prominence=prominence)
            half_widths = peak_widths(counts, peaks)[0] * bin_width
            for x, y, half_width in zip(bin_centers[peaks], counts[peaks], half_widths):
                plt.errorbar(
                    [x],
                    [y],
                    xerr=[half_width],
                    c="black",
                    fmt="",
                    capsize=2,
                    elinewidth=0.2,
                    capthick=0.2,
                )
                plt.annotate(
                    f"{x:.03f} ± {half_width:.03f} MeV",
                    (
                        x,
                        y
                        + height_scale * (0.001 * text_size)
                        + (np.random.random() - 0.5) * rng_weight,
                    ),
                    ha="left",
                    fontsize=text_size,
                )

    plt.xlabel(xlabel)
    plt.ylabel("Count per Day")
    # plt.ylim(ylims)
    plt.title(title)
    if legend is not None:
        plt.legend(legend)

    plt.show()

# Loaders

In [ ]:
keys = [
    "E",
    "p_x",
    "p_y",
    "p_z",
    "x",
    "y",
    "z",
    "creator",
    "time",
    "source"
]


def decode(b):
    return b.decode("utf-8")


decode_v = np.vectorize(decode)


def load_hdf5(path):
    dfs = []
    with h5py.File(path, "r") as f:
        for key in keys:
            data = f["default_ntuples"]["SecondaryGammas"][key]["pages"][()]
            if isinstance(data[0], bytes):
                data = decode_v(data)

            dfs.append(pd.DataFrame(data, columns=[key]))

    return pd.concat(dfs, axis=1)


def load_csv(path):
    df = pd.read_csv(path, comment="#", header=None)
    df.columns = keys.copy()
    print(f"{df.shape[0]:,} gamma rays recorded")
    return df


def load_raw(path):
    rows = []
    with open(path, "rb") as file:
        while data := file.read(3 * 8):
            energy, p_z, depth = struct.unpack("ddd", data)
            rows.append([energy, p_z, depth])
    df = pd.DataFrame(rows, columns=["E", "p_z", "z"])
    return df

In [ ]:
df = load_csv("finals/big-info.csv")
exposure = 0.0155408565 #days
# exposure = 1 # days
df

# Analysis

## Masking Utilities

In [ ]:
property_range = lambda property, lower, upper, mask=None: (
    df if mask is None else df[mask]
)[property].between(lower, upper)
energy_range = lambda lower, upper, mask=None: property_range("E", lower, upper, mask)
depth_range = lambda lower, upper, mask=None: property_range("z", lower, upper, mask)
include_processes = lambda names, mask=None: (df if mask is None else df[mask])[
    "creator"
].isin(names)
exclude_processes = lambda names, mask=None: ~(df if mask is None else df[mask])[
    "creator"
].isin(names)
include_sources = lambda names, mask=None: (df if mask is None else df[mask])[
    "source"
].isin(names)
exclude_sources = lambda names, mask=None: ~(df if mask is None else df[mask])[
    "source"
].isin(names)
upward = df["p_z"] > 0.0

In [ ]:
process_names, process_counts = np.unique(df["creator"], return_counts=True)
process_counts, process_names = zip(*sorted(zip(process_counts, process_names))[::-1])
source_names, source_counts = np.unique(df["source"], return_counts=True)
source_counts, source_names = zip(*sorted(zip(source_counts, source_names))[::-1])

In [ ]:
plot_with_peaks(df["z"], n_bins=512)

In [ ]:
df[include_processes(["", "neutronInelastic"])]["z"].agg(["mean", "sem"])

## Energy Spectrum

In [ ]:
def plot_energy_spectrum(
    e_low, e_high, n_bins=512, prominence=None, clip=None, text_size=5
):
    energies = lambda procs: df[energy_range(e_low, e_high) & include_processes(procs)][
        "E"
    ]
    plot_with_peaks(
        data=[energies([proc]) for proc in process_names[:4]]
        + [energies(process_names[4:])],
        n_bins=n_bins,
        # prominence=prominence,
        xlabel="Energy (MeV)",
        title="Gamma Ray Energies at Creation in Soil",
        clip_counts=clip,
        text_size=text_size,
        divide=exposure,
        stacked=True,
        bin_range=(e_low, e_high),
        legend=list(process_names[:4]) + ["other"],
    )


# plot_energy_spectrum(
#     2.0, 5.0, clip=500000, n_bins=2048, prominence=100000, text_size=10
# )
# plot_energy_spectrum(0.5, 0.6, n_bins=256, prominence=200000, text_size=10)

annihil_sources, annihil_counts = np.unique(df[energy_range(0.510, 0.512)]["source"], return_counts=True)

plt.bar(annihil_sources, annihil_counts / annihil_counts.sum())

for source, prop in zip(annihil_sources, annihil_counts / annihil_counts.sum()):
    print(f"{source}: {prop:%}")

In [ ]:
mask_4438 = energy_range(4.4375, 4.4385)
mask_4439 = energy_range(4.4385, 4.4395)

In [ ]:
plt.figure(figsize=(10, 2))
plt.yscale("log")
plot_with_peaks(
    df[energy_range(2.0, 10.0)]["E"],
    1000,
    # prominence=30000,
    xlabel="Energy (MeV)",
    divide=exposure,
    # text_size=3,
    title="Energies of Gamma Rays Produced in Soil",
    # rng_weight=400000.0
)
# plt.annotate("2.223 MeV", (2.223, 10e6))
plt.show()

In [ ]:
plot_with_peaks(
    [df[mask_4438]["E"], df[mask_4439]["E"]],
    16,
    stacked=True,
    bin_range=(4.43, 4.44),
)
# df[""]

## Depth Profiles

In [ ]:
# plt.yscale("log")

plt.figure(figsize=(10, 2))
plt.yscale("log")
# plot_with_peaks(
#     df[energy_range(2.0, 10.0)]["E"],
#     50000,
#     prominence=30000,
#     xlabel="Energy of Gamma Ray (MeV)",
#     divide=exposure,
#     text_size=3,
#     title="Energies of Gamma Rays Produced in Soil",
#     rng_weight=400000.0
# )

name_dict = {
    "annihil": "Annihilation",
    "neutronInelastic": "Inelastic Neutron Scattering",
    "nCapture": "Neutron Capture",
    "RadioactiveDecay": "Radioactive Decay",
    "protonInelastic": "Inelastic Proton Scattering",
    "eBrem": "Bremsstrahlung",
    "gamma": "Gamma",
    "muMinusCaptureAtRest": "$\\mu^{-}$ Capture at Rest",
}

plot_with_peaks(
    data=[
        df[energy_range(2.0, 10.0) & exclude_sources(["neutron"])]["E"],
        df[energy_range(2.0, 10.0) & include_sources(["neutron"])]["E"],
    ],
    n_bins=1000,
    xlabel="Energy (MeV)",
    title="Energies of Gamma Rays Produced in Soil, by Source Cosmic Ray Particle",
    xlims=(2.0, 10.0),
    stacked=True,
    legend=["All Others", "Neutrons"],
    bin_range=(2.0, 10.0),
    divide=exposure,
    text_size=3,
)

In [ ]:
# plt.yscale("log")

plot_with_peaks(
    data=[df[include_processes([name])]["z"] / 10 for name in process_names[1:6]]
    + [df[exclude_processes(process_names[1:6])]["z"] / 10],
    n_bins=256,
    xlabel="Depth in Soil (cm)",
    title="Depth of Gamma Rays Produced in Soil at Time of Production",
    xlims=(-50, 0),
    stacked=True,
    legend=list(name_dict[name] for name in process_names[1:6]) + ["Bremsstrahlung"],
    bin_range=(-50, 0),
    divide=exposure,
)

In [ ]:
print(df[include_processes(["nCapture"])]["z"].agg(["mean", "std", "sem"]))

# plt.yscale("log")
plt.ylim(0, 150000)
plot_with_peaks(
    data=[df[exclude_processes([]) & include_sources([name])]["z"] / 10 for name in source_names[:]],
    # + [df[exclude_processes(process_names[:8])]["z"] / 10],
    n_bins=256,
    xlabel="Depth (cm)",
    title="Depth of Non-Primary Gamma Rays at Creation, by Original Source",
    xlims=(-50, 0),
    stacked=True,
    legend=list(source_names[:]),
    bin_range=(-50, 0),
)

In [ ]:
print(dict(zip(*np.unique(df[energy_range(1.77899, 1.779)]["creator"], return_counts=True))))
plot_with_peaks(df[energy_range(1.77899, 1.779)]["E"], n_bins=256, divide=exposure)

In [ ]:
mctaggart = include_processes(["neutronInelastic", "nCapture", "RadioactiveDecay"])
print(df[mctaggart]["z"].aggregate(["mean", "median", "std"]) / 10)
plt.yscale("log")
plot_with_peaks(
    df[mctaggart]["z"] / 10, 100, title="Depth of Gamma Ray Production (McTaggart Only)"
)

## Creation Processes

In [ ]:
plt.yscale("log")
plt.bar(process_names, process_counts)
plt.xticks(rotation="vertical")
plt.title("Creator Processes")
plt.show()

In [ ]:
def process_breakdown(df):
    names, counts = np.unique(df["creator"], return_counts=True)
    counts, names = zip(*sorted(zip(counts, names))[::-1])
    counts = np.array(counts)
    uncerts = np.sqrt(counts)
    total_count = counts.sum()
    for name, pc, pc_u in zip(names, counts / total_count, uncerts / total_count):
        print(f"{name} & {100*pc:.04f}\\% & {100*pc_u:.04f}\\% \\\\")

process_breakdown(df[df["source"] == "neutron"])
print()
process_breakdown(df)

In [ ]:
print(
    f"{100*df[include_processes(["eBrem"])].size / df.size:.1f}% of gamma rays generated by Bremsstrahlung radiation"
)

## Time of Creation

In [ ]:
# plt.yscale("log")
mask = lambda name: include_processes([name])
plot_with_peaks(
    data=[np.log10(df[mask(name)]["time"]) for name in process_names[2:7]]
    + [np.log10(df[exclude_processes(process_names[:7])]["time"])],
    n_bins=256,
    title="Time of Creation of Gamma Rays in Soil\n(excluding Bremsstrahlung & Annihilation)",
    xlabel="$\\log_{10}$ Time (log ns)",
    stacked=True,
    legend=list(name_dict[name] for name in process_names[2:7]) + ["other"],
    bin_range=(-3.0, 16.0),
    xlims=(-3.0, 16.0),
    divide=exposure
)

num = df[include_processes(["annihil", "eBrem"])].size
tot = df.size
num_u = np.sqrt(num)
print(f"{100 * num/tot:.03f}% +/- {100 * num_u / tot:.03f}%")

## Sources

In [ ]:
plt.yscale("log")
plt.bar(source_names, source_counts)
plt.xticks(rotation="vertical")
plt.title("Primary Sources")
plt.show()
source_dict = {k: v for k, v in zip(source_names, source_counts)}
source_dict["neutron"]

## Isotropy/Anisotropy

In [ ]:
plot_with_peaks(
    [
        np.atan2(
            df[include_processes([name])]["p_x"], df[include_processes([name])]["p_y"]
        )
        for name in process_names[:8]
    ]
    + [
        np.atan2(
            df[exclude_processes(process_names[:8])]["p_x"],
            df[exclude_processes(process_names[:8])]["p_y"],
        )
    ],
    512,
    xlabel="Azimuthal Angle, $\\varphi$",
    stacked=True,
    legend=list(process_names[:8]) + ["other"],
    bin_range=(-np.pi, np.pi),
)

In [ ]:
from uncertainties import ufloat
annihil_up = df[include_processes(["annihil"]) & (df["p_z"] > 0.0)].size
annihil_tot = df[include_processes(["annihil"])].size
print(f"{ufloat(annihil_up, np.sqrt(annihil_up)) / annihil_tot:%}")

plot_with_peaks(
    df[include_processes(["eBrem"])]["p_z"],
    512,
    xlabel="Momentum Direction in $z$, $p_z$",
    title="eBrem $p_z$ Bias",
)
plot_with_peaks(
    df[include_processes(["annihil"])]["p_z"],
    512,
    xlabel="Momentum Direction in $z$, $p_z$",
    title="annihil $p_z$ Bias",
)
# plot_with_peaks(
#     df[energy_range(4.437, 4.439)]["p_z"],
#     16,
#     xlabel="Momentum Direction in $z$, $p_z$",
#     title="Carbon peak $p_z$ Bias",
# )
# plot_with_peaks(
#     [df[include_processes([name])]["p_z"] for name in process_names[2:8]],
#     512,
#     xlabel="Momentum Direction in $z$, $p_z$",
#     stacked=True,
#     legend=process_names[2:8],
#     bin_range=(-1.0, 1.0),
# )

# n_down = df[include_processes(["eBrem"]) & (df["p_z"] < 0.0)].size
# n_tot = df[include_processes(["eBrem"])].size/
# n_down_u = np.sqrt(n_down)
# print(f"{100 * n_down/n_tot:.03f}% +/- {100 * n_down_u/n_tot:.03f}%")

stats, bin_edges, _ = binned_statistic(
    df[energy_range(2.0, 10.0) & include_processes(["eBrem"])]["E"],
    (df[energy_range(2.0, 10.0) & include_processes(["eBrem"])]["p_z"] < 0).astype(float),
    bins=32,
)

plt.stairs(stats, bin_edges, fill=True)

In [ ]:
heatmap, x_edges, y_edges = np.histogram2d(df["p_z"], df["z"] / 10.0, bins=10)
extent = (x_edges[0], x_edges[-1], y_edges[0], y_edges[-1])
e_range = df["E"].between(2.0, 7.0)
isotropic = exclude_processes(["eBrem", "annihil"])
anisotropic = include_processes(["eBrem", "annihil"])

plt.hexbin(
    df[isotropic]["p_z"], df[isotropic]["z"] / 10.0, linewidths=0.2, norm=LogNorm()
)
plt.colorbar()
plt.xlabel("$z$-Momentum Bias, $p_z$")
plt.ylabel("Depth, $z$ (cm)")
plt.title("Heatmap of $z$-Momentum Bias vs. Depth\n(no eBrem or annihil)")
plt.show()

plt.hexbin(
    df[anisotropic]["p_z"], df[anisotropic]["z"] / 10.0, linewidths=0.2, norm=LogNorm()
)
plt.colorbar()
plt.xlabel("$z$-Momentum Bias, $p_z$")
plt.ylabel("Depth, $z$ (cm)")
plt.title("Heatmap of $z$-Momentum Bias vs. Depth\n(only eBrem and annihil)")
plt.show()

In [ ]:
mask = df["E"].between(0, 10) & isotropic
plt.hexbin(df[mask]["E"], df[mask]["z"] / 1e3, linewidths=0.2, norm=LogNorm())
plt.colorbar()
plt.xlabel("Energy, $E$ (MeV)")
plt.ylabel("Depth, $z$ (m)")
plt.title(
    "Number of Gamma Rays Produced Below 10 MeV\n(excl. Bremsstrahlung & Annihilation)"
)
plt.show()

In [ ]:
mask = df["E"].between(0, 10)
plt.hexbin(df[mask]["p_z"], df[mask]["E"], linewidths=0.2, norm=LogNorm())
plt.colorbar(label="Count")
plt.xlabel("Momentum Angle Cosine, $\\cos \\theta$")
plt.ylabel("Energy, $E$ (MeV)")
plt.title("Number of Gamma Rays Produced Below 10 MeV")
plt.show()

In [ ]:
def plot_process(name, plot_type: Literal["prop", "num"] = "prop", mask=None):
    plt.hexbin(
        (df if mask is None else df[mask])["p_z"],
        0.1 * (df if mask is None else df[mask])["z"],
        linewidths=0.2,
        C=include_processes([name], mask),
        reduce_C_function=np.mean if plot_type == "prop" else np.sum,
    )
    type_str = "Proportion" if plot_type == "prop" else "Number"
    plt.colorbar(label=type_str)
    plt.xlabel("$z$-Momentum Bias, $p_z$")
    plt.ylabel("Depth, $z$ (cm)")
    plt.title(f"{type_str} of Gamma Rays Produced\nby {name}")
    plt.show()

In [ ]:
plot_process(
    "eBrem",
    plot_type="prop",
    mask=df["E"].between(3, 1e10) & df["z"].between(-10000, 0),
)